# Welcome to the Local Particle Filter (LPF) tutorial


## Prerequisites

<u>This tutorial requires completion of the **0.qg_tutorial_start** tutorial</u>. We will use the truth run generated from that tutorial. 



## Introduction to LPF
xxxxx


We now set environment variables and verify that required executables exist. **You need to do this anytime you open this notebook for a new localhost session**

In [1]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
export JEDI_BIN_v2=/home/nonroot/build-v2/bin
export JEDI_EDU_v2=/home/nonroot/shared-v2/EDU

In [2]:
# Verify LPF executables exist
ls $JEDI_BIN_v2/qg_gen_ens_pert_B.x
ls $JEDI_BIN_v2/qg_etkf.x
ls $JEDI_BIN_v2/qg_ens_mean_variance.x

/home/nonroot/build-v2/bin/qg_gen_ens_pert_B.x
/home/nonroot/build-v2/bin/qg_etkf.x
/home/nonroot/build-v2/bin/qg_ens_mean_variance.x



### Table of LPF Experiments

| Experiment Name | letkf yaml file | Description |
| --- | --- | --- |
| exp_default | lpf.yaml | Default single-observation LPF experiment
| exp_dbl_loc | lpf_dbl_loc.yaml | Double the localization scale for LPF
| exp_5mem | lpf_5mem.yaml | Reduces ensemble size to 5
| exp_mult_obs  | lpf_mult_obs.yaml | Assimilates 100 observations


# Experiment 1: LPF with one observation
***

### Step 1 & 2: Verify that truth and background ensemble exist

Double-check the  files exist from the qgstart tutorial. If not, please run that tutorial first!

In [3]:
ls $JEDI_EDU/qgstart/output/truth

truth.fc.2009-12-15T00:00:00Z.P10D.nc
truth.fc.2009-12-15T00:00:00Z.P10DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P10DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P10DT6H.nc
truth.fc.2009-12-15T00:00:00Z.P11D.nc
truth.fc.2009-12-15T00:00:00Z.P11DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P11DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P11DT6H.nc
truth.fc.2009-12-15T00:00:00Z.P12D.nc
truth.fc.2009-12-15T00:00:00Z.P12DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P12DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P12DT6H.nc
truth.fc.2009-12-15T00:00:00Z.P13D.nc
truth.fc.2009-12-15T00:00:00Z.P13DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P13DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P13DT6H.nc
truth.fc.2009-12-15T00:00:00Z.P14D.nc
truth.fc.2009-12-15T00:00:00Z.P14DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P14DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P14DT6H.nc
truth.fc.2009-12-15T00:00:00Z.P15D.nc
truth.fc.2009-12-15T00:00:00Z.P15DT12H.nc
truth.fc.2009-12-15T00:00:00Z.P15DT18H.nc
truth.fc.2009-12-15T00:00:00Z.P15DT6H.nc
truth.fc.2009-12-15T00

In [4]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/bg/bkgd.ens.*P1D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Background files don't exist! Please see the 0.qg_tutorial_start tutorial to generate the background ensemble"
fi

/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.10.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.11.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.12.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.13.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.14.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.15.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.16.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.17.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.18.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.19.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.2.2009-12-30T00:00:00Z.P1D.nc
/home/

### Step 3: Verify observation file

In [5]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Observation file doesn't exist! Please see the 0.qg_tutorial_start tutorial to generate the single observation"
fi

/home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc


To verify the observation is correct, run the following python script and cat command to view the observation location and values. Note the $lon$, $lat$ should be $[150]$ and $[50]$, respectively, close to the northeastern part of our plot domain, the same as the dot shown in the above variance figure. 

In [6]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_default/plots/qg_oneob

Parameters:
 - model: qg
 - diagnostic: obs
 - filepath: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/plots/qg_oneob
Run script
 -> Observations values written in /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/plots/qg_oneob.txt


In [7]:
cat $JEDI_EDU_v2/qgLPF/output/exp_default/plots/qg_oneob.txt

# location / value / hofx
[ 150.   50. 7000.] / [-2.86101159e+08] / [-2.87489617e+08]


We can now run the LPF!

### Step 4: Run the LPF!

```yaml

time window:
  begin: &date_bgn 2009-12-30T18:00:00Z
  length: PT12H

local ensemble DA: 
  solver: PF  # Controls which flavor of ensemble solver is used
  inflation:    # Covariance inflation (applied after LETKF update)
    mult: 1.0   # Multiplicative inflation factor
  pf frac neff: 0.8

obs space:
  obsdatain:
    obsfile: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
  obsdataout:
    obsfile: qgLPF/output/exp_default/obs/lpf_oneob.obs3d.nc
  obs type: Stream

In [9]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf.yaml 

OOPS Starting 2026-04-24 01:49:02 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/lpf.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/lpf.yaml, root={time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {members from template => {template => {states => ({date => 2009-12-31T00:00:00Z , filename => /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc})} , pattern => %mem% , nmembers => 20}} , observations => {observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_default/plots/lpf_oneob.obs3d.nc} , obs type => Stream} , obs error => {covariance model => diagonal} , obs localizations => ({lengthscale => 4e+06})})} , driver => {update obs config with geometry info => false , save prior mean => true , sa

In [10]:
ls $JEDI_EDU_v2/qgLPF/output/exp_default/da

lpf.000000.an.2009-12-31T00:00:00Z.nc  lpf.000012.an.2009-12-31T00:00:00Z.nc
lpf.000001.an.2009-12-31T00:00:00Z.nc  lpf.000013.an.2009-12-31T00:00:00Z.nc
lpf.000002.an.2009-12-31T00:00:00Z.nc  lpf.000014.an.2009-12-31T00:00:00Z.nc
lpf.000003.an.2009-12-31T00:00:00Z.nc  lpf.000015.an.2009-12-31T00:00:00Z.nc
lpf.000004.an.2009-12-31T00:00:00Z.nc  lpf.000016.an.2009-12-31T00:00:00Z.nc
lpf.000005.an.2009-12-31T00:00:00Z.nc  lpf.000017.an.2009-12-31T00:00:00Z.nc
lpf.000006.an.2009-12-31T00:00:00Z.nc  lpf.000018.an.2009-12-31T00:00:00Z.nc
lpf.000007.an.2009-12-31T00:00:00Z.nc  lpf.000019.an.2009-12-31T00:00:00Z.nc
lpf.000008.an.2009-12-31T00:00:00Z.nc  lpf.000020.an.2009-12-31T00:00:00Z.nc
lpf.000009.an.2009-12-31T00:00:00Z.nc  lpf_meaninc.an.2009-12-31T00:00:00Z.nc
lpf.000010.an.2009-12-31T00:00:00Z.nc  lpf_meanprior.an.2009-12-31T00:00:00Z.nc
lpf.000011.an.2009-12-31T00:00:00Z.nc


**One important note for the file list above. The posterior analysis ensemble mean is the "000000" ensemble member.** 

We will now plot the results using the python plotting script. Let's examine the mean increments. That is, the mean ensemble analysis minus the mean ensemble background. (Alternatively, you could plot the contents of the `da/lpf_meaninc.an.2009-12-31T00:00:00Z.nc` file directly)

In [11]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_default/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU_v2/qgLPF/output/exp_default/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU_v2/qgLPF/output/exp_default/plots/lpf_mean_increments --title "ens. mean anl inc."

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/da/lpf.000000.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc
 - fieldmax: 50000000.0
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
 - plothofx: False
 - plotwind: True
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/plots/lpf_mean_increments
 - title: ens. mean anl inc.
Run script
['/home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/da/lpf.000000.an.2009-12-31T00:00:00Z.nc']
lon,lat,val of first ob = 150.0,50.0,1388457.7714409828
Variable x, RMSD = 3631794.1678472743
data range:(-37979757.86018881,737451.5581282377), plot range: (-50000000.0,50000000.0) 
 -> plot produced: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_default/plots/lpf_mean_increments_x_diff.jpg
lon,lat,

#### Ensemble Mean Increments 
| LETKF | LPF |
|---|---|
| ![](./images/ensmean_inc_default_x_v2.jpg) | ![](./images/lpf_mean_increments_x_diff.jpg) |




# Experiment 2: Change the horizontal localization length scale

***

### Steps 1-3: Truth, Background Ensemble, Observations

This step can be skipped, as everything remains the same. 

### Step 4: Run the data assimilation:
Let's compare `lpf_dbl_loc.yaml` to the default experiment `lpf.yaml`:

> **Note on diff command<br>**
> The diff command compares two files and prints out lines where  differences are found. The "<" and ">" point to which file the line is from (in this case, "<" is for lpf.yaml, and ">" is for lpf_dbl_loc.yaml.<br>
> If the files are different, the diff command return a code of 1. The python notebook shows this as an error in a red box after execution; however, for the diff command it can be safely ignored, since it is a normal flag to indicate differences were found (the return code is 0 if no differences were found)


In [13]:
diff $JEDI_EDU_v2/qgLPF/yamls/lpf.yaml $JEDI_EDU_v2/qgLPF/yamls/lpf_dbl_loc.yaml

27c27
<         obsfile: qgLPF/output/exp_default/plots/lpf_oneob.obs3d.nc
---
>         obsfile: qgLPF/output/exp_dbl_loc/plots/lpf_oneob.obs3d.nc
32c32
<     - lengthscale: 4.0e6                # Note no vertical localization is applied 
---
>     - lengthscale: 8.0e6                # Note no vertical localization is applied 
48c48
<   - datadir: qgLPF/output/exp_default/da
---
>   - datadir: qgLPF/output/exp_dbl_loc/da
54c54
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_dbl_loc/da
60c60
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_dbl_loc/da


: 1

The main parameter that changes the length scale for localization is `lengthscale`, which is doubled to 8.0e6 (i.e. 8000 km). There are also changes to the output file locations for the analysis that replaced  `exp_default` by `exp_dbl_loc`, ensuring we do not overwrite any previous results.

You may run the LPF:

In [14]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_dbl_loc.yaml

OOPS Starting 2026-04-24 01:51:52 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/lpf_dbl_loc.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/lpf_dbl_loc.yaml, root={time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {members from template => {template => {states => ({date => 2009-12-31T00:00:00Z , filename => /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc})} , pattern => %mem% , nmembers => 20}} , observations => {observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_dbl_loc/plots/lpf_oneob.obs3d.nc} , obs type => Stream} , obs error => {covariance model => diagonal} , obs localizations => ({lengthscale => 8e+06})})} , driver => {update obs config with geometry info => false , save prior m

In [15]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_dbl_loc/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU_v2/qgLPF/output/exp_dbl_loc/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU_v2/qgLPF/output/exp_dbl_loc/plots/lpf_mean_increments --title "ens. mean anl inc."

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_dbl_loc/da/lpf.000000.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_dbl_loc/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc
 - fieldmax: 50000000.0
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
 - plothofx: False
 - plotwind: True
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_dbl_loc/plots/lpf_mean_increments
 - title: ens. mean anl inc.
Run script
['/home/nonroot/shared-v2/EDU/qgLPF/output/exp_dbl_loc/da/lpf.000000.an.2009-12-31T00:00:00Z.nc']
lon,lat,val of first ob = 150.0,50.0,1388457.7714409828
Variable x, RMSD = 5751384.9415842695
data range:(-41544993.364757836,4049606.497684896), plot range: (-50000000.0,50000000.0) 
 -> plot produced: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_dbl_loc/plots/lpf_mean_increments_x_diff.jpg
lon,lat

#### Ensemble Mean Increments - Doubled Loc
| LETKF | LPF |
|---|---|
| ![](./images/ensmean_inc_dbl_loc_v2.jpg) | ![](./images/lpf_mean_increments_x_diff_dbl_loc.jpg) |


# Experiment 3: LPF with reduced ensemble members

### Steps 1-3: Truth, Background Ensemble, Observations

As in experiment 2, these files remain the same

### Step 4: Run LPF
Let us compare `lpf_5mem.yaml` to the default experiment `lpf.yaml`. 

In [17]:
diff $JEDI_EDU_v2/qgLPF/yamls/lpf.yaml $JEDI_EDU_v2/qgLPF/yamls/lpf_5mem.yaml

17c17
<     nmembers: 20                       # Number of ensemble members to analyze
---
>     nmembers: 5                       # Number of ensemble members to analyze
27c27
<         obsfile: qgLPF/output/exp_default/plots/lpf_oneob.obs3d.nc
---
>         obsfile: qgLPF/output/exp_5mem/plots/lpf_oneob_newloc.obs3d.nc
32c32
<     - lengthscale: 4.0e6                # Note no vertical localization is applied 
---
>     - lengthscale: 8.0e6                # Note no vertical localization is applied 
48c48
<   - datadir: qgLPF/output/exp_default/da
---
>   - datadir: qgLPF/output/exp_5mem/da
54c54
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_5mem/da
60c60
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_5mem/da


: 1

In [18]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_5mem.yaml 

OOPS Starting 2026-04-24 01:53:50 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/lpf_5mem.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/lpf_5mem.yaml, root={time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {members from template => {template => {states => ({date => 2009-12-31T00:00:00Z , filename => /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc})} , pattern => %mem% , nmembers => 5}} , observations => {observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_5mem/plots/lpf_oneob_newloc.obs3d.nc} , obs type => Stream} , obs error => {covariance model => diagonal} , obs localizations => ({lengthscale => 8e+06})})} , driver => {update obs config with geometry info => false , save prior mean

In [19]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_5mem/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU_v2/qgLPF/output/exp_5mem/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc  \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU_v2/qgLPF/output/exp_5mem/plots/lpf_mean_increments --title "ens. mean anl inc."

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_5mem/da/lpf.000000.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_5mem/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc
 - fieldmax: 50000000.0
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
 - plothofx: False
 - plotwind: True
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_5mem/plots/lpf_mean_increments
 - title: ens. mean anl inc.
Run script
['/home/nonroot/shared-v2/EDU/qgLPF/output/exp_5mem/da/lpf.000000.an.2009-12-31T00:00:00Z.nc']
lon,lat,val of first ob = 150.0,50.0,1388457.7714409828
Variable x, RMSD = 2521318.8379094903
data range:(-14290169.34084773,14503973.490549214), plot range: (-50000000.0,50000000.0) 
 -> plot produced: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_5mem/plots/lpf_mean_increments_x_diff.jpg
lon,lat,val of first o

#### Ensemble Mean Increments - LPF
| LPF 20 members | LPF 5 members |
|---|---|
| ![](./images/lpf_mean_increments_x_diff_dbl_loc.jpg) | ![](./images/lpf_mean_increments_x_diff_5mem.jpg) |





# Experiment 4: Assimilate multiple observations
***

We now embark on a more realistic situation in which 100 observations are assimilated all at once.

### Steps 1-3: Truth, Background Ensemble, Observations

The truth and background ensemble remain unchanged. We will copy the 100-observations file produced from the **0.qg_tutorial_start** tutorial  to the **exp_mult_obs** folder 


In [20]:
cp $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs
ls $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs

hofx_meananl.obs3d.nc  lpf_100obs.obs3d.nc
hofx_meanbkg.obs3d.nc  truth_100obs.obs3d.nc


### Step 4: Run the LPF

Let's compare the 100-obs settings for `lpf_mult_obs.yaml` to the single-observation settings in `lpf.yaml`

In [22]:
diff $JEDI_EDU_v2/qgLPF/yamls/lpf.yaml $JEDI_EDU_v2/qgLPF/yamls/lpf_mult_obs.yaml

25c25
<         obsfile: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
---
>         obsfile: qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc
27c27
<         obsfile: qgLPF/output/exp_default/plots/lpf_oneob.obs3d.nc
---
>         obsfile: qgLPF/output/exp_mult_obs/obs/lpf_100obs.obs3d.nc
32a33
>                                         
48c49
<   - datadir: qgLPF/output/exp_default/da
---
>   - datadir: qgLPF/output/exp_mult_obs/da
54c55
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_mult_obs/da
60c61
<   datadir: qgLPF/output/exp_default/da
---
>   datadir: qgLPF/output/exp_mult_obs/da


: 1

In [23]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_mult_obs.yaml 

OOPS Starting 2026-04-24 01:56:29 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/lpf_mult_obs.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/lpf_mult_obs.yaml, root={time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {members from template => {template => {states => ({date => 2009-12-31T00:00:00Z , filename => /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc})} , pattern => %mem% , nmembers => 20}} , observations => {observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_mult_obs/obs/lpf_100obs.obs3d.nc} , obs type => Stream} , obs error => {covariance model => diagonal} , obs localizations => ({lengthscale => 4e+06})})} , driver => {update obs config with geometry info => false , save prior mean => true

<br><br> You can see the results, after thinking about your expectations, via:

In [24]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        --plotObsLocations $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/lpf_meaninc --title "analysis increment ENSMEAN"

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc
 - fieldmax: None
 - plotObsLocations: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc
 - plotObsValues: None
 - plothofx: False
 - plotwind: False
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/lpf_meaninc
 - title: analysis increment ENSMEAN
Run script
['/home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00:00:00Z.nc']
Obs_type=Stream
res.groups={'Stream': <class 'netCDF4.Group'>
group /Stream:
    dimensions(sizes): nobs(100)
    variables(dimensions): |S1 times(nobs, nstrmax)
    groups: Location, ObsError, EffectiveQC, hofx, EffectiveError, ObsBias, ObsValue}
Variable x, RMSD = 13937259.349339347



| LETKF | LPF |
|---|---|
| ![](./images/letkf_meaninc_x_100obs_v2.jpg) | ![](./images/lpf_meaninc_x_diff_100obs.jpg) |



In [25]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/lpf_meananl_error --title "mean analysis error" 
python ./plot.py qg fields \
        $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/lpf_meanbkg_error --title "mean background error" 

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
 - fieldmax: None
 - plotObsLocations: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc
 - plotObsValues: None
 - plothofx: False
 - plotwind: False
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/lpf_meananl_error
 - title: mean analysis error
Run script
['/home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00:00:00Z.nc']
Obs_type=Stream
res.groups={'Stream': <class 'netCDF4.Group'>
group /Stream:
    dimensions(sizes): nobs(100)
    variables(dimensions): |S1 times(nobs, nstrmax)
    groups: Location, ObsError, EffectiveQC, hofx, EffectiveError, ObsBias, ObsValue}
Variable x, RMSD = 11254144.062979965
data range:(-31

## Stream function

### LETKF
| | |
|---|---|
| ![](./images/letkf_meanbkg_error_x_100obs_v2.jpg) | ![](./images/letkf_meananl_error_x_100obs_v2.jpg) |

### LPF
| | |
|---|---|
| ![](./images/lpf_meanbkg_error_x_diff_100obs.jpg) | ![](./images/lpf_meananl_error_x_diff_100obs.jpg) |



## Potential vorticity 

### LETKF
| | |
|---|---|
| ![](./images/letkf_meanbkg_error_q_100obs_v2.jpg) | ![](./images/letkf_meananl_error_q_100obs_v2.jpg) |

### LPF
| | |
|---|---|
| ![](./images/lpf_meanbkg_error_q_diff_100obs.jpg) | ![](./images/lpf_meananl_error_q_diff_100obs.jpg) |

In [26]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_hofx3d.x ./qgLPF/yamls/hofx_mult_obs.yaml

OOPS Starting 2026-04-24 02:35:38 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/hofx_mult_obs.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/hofx_mult_obs.yaml, root={geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , state => {date => 2009-12-31T00:00:00Z , filename => qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc} , time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , observations => {get values => {variable change => {input variables => () , output variables => ()}} , observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_mult_obs/obs/hofx_meanbkg.obs3d.nc} , obs type => Stream} , get values => {interpolation type => default_1}})}}]
OOPS_STATS ObjectCountHelper started.
OOPS_STATS Run start                                - Runtime:      0.03 sec,  Memory: total:    35.85 MB, per

In [27]:
cd $JEDI_EDU_v2
$JEDI_BIN_v2/qg_hofx3d.x ./qgLPF/yamls/hofx_mult_obs_an.yaml

OOPS Starting 2026-04-24 02:35:44 (UTC+0000)
Configuration input file is: ./qgLPF/yamls/hofx_mult_obs_an.yaml
Full configuration is:YAMLConfiguration[path=./qgLPF/yamls/hofx_mult_obs_an.yaml, root={geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , state => {date => 2009-12-31T00:00:00Z , filename => qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00:00:00Z.nc} , time window => {begin => 2009-12-30T18:00:00Z , length => PT12H} , observations => {get values => {variable change => {input variables => () , output variables => ()}} , observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {obsfile => qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc} , obsdataout => {obsfile => qgLPF/output/exp_mult_obs/obs/hofx_meananl.obs3d.nc} , obs type => Stream} , get values => {interpolation type => default_1}})}}]
OOPS_STATS ObjectCountHelper started.
OOPS_STATS Run start                                - Runtime:      0.03 sec,  Memory: total:    35.35 MB, 

This results in two files in `exp_mult_obs/obs` directory:, `hofx_meananl.obs3d.nc` and `hofx_meanbkg.obs3d.nc`. We can extract the contents of these observation files by using the python script:

In [28]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs/hofx_meananl.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/hofx_meananl
cat $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/hofx_meananl.txt

Parameters:
 - model: qg
 - diagnostic: obs
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/obs/hofx_meananl.obs3d.nc
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meananl
Run script
 -> Observations values written in /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meananl.txt
# location / value / hofx
[ -29.87208056    3.63767342 3266.44902118] / [3115105.01660962] / [562261.92477709]
[ 178.98653093    8.23197272 5786.33931931] / [9341113.13679803] / [11288975.19128204]
[  79.31681614   59.17619073 5270.58105916] / [-1.75884984e+08] / [-1.76465324e+08]
[155.72065003  78.38089478  90.07995483] / [-93802685.12679093] / [-82999391.16377693]
[-179.95882281   18.17969161 8859.42095192] / [-26903573.09265] / [-35882371.29531529]
[-133.8751988    77.90727735 7090.42522358] / [-3.60603479e+08] / [-3.68648564e+08]
[ -71.16027561   22.64921372 3572.69760454] / [-78510060.90002471] / [-79481365.76557425]
[ 179.65458554   25.485879

In [29]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/obs/hofx_meanbkg.obs3d.nc \
        --output $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/hofx_meanbkg
cat $JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots/hofx_meanbkg.txt

Parameters:
 - model: qg
 - diagnostic: obs
 - filepath: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/obs/hofx_meanbkg.obs3d.nc
 - output: /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meanbkg
Run script
 -> Observations values written in /home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meanbkg.txt
# location / value / hofx
[ -29.87208056    3.63767342 3266.44902118] / [3115105.01660962] / [2081698.46855204]
[ 178.98653093    8.23197272 5786.33931931] / [9341113.13679803] / [5292943.07519392]
[  79.31681614   59.17619073 5270.58105916] / [-1.75884984e+08] / [-1.79195139e+08]
[155.72065003  78.38089478  90.07995483] / [-93802685.12679093] / [-66958035.72184688]
[-179.95882281   18.17969161 8859.42095192] / [-26903573.09265] / [-41574752.6843902]
[-133.8751988    77.90727735 7090.42522358] / [-3.60603479e+08] / [-3.61221426e+08]
[ -71.16027561   22.64921372 3572.69760454] / [-78510060.90002471] / [-60450178.20809872]
[ 179.65458554   25.4858792



In this tutorial, we will introduce computation of statistics, namely **Root-mean-square error (RMSE).**  

Here, the script `computermsd.bash` is provided within the `plot_scripts` directory to compute and print out this RMSE statistic for you. We need to provide it the text files produced above to run:


In [30]:
cd $JEDI_EDU/plots_scripts
PLOT_DIR_v2=$JEDI_EDU_v2/qgLPF/output/exp_mult_obs/plots
# To run: ./computermsd.bash <hofx_background_file.txt> <hofx_analysis_file.txt>
./computermsd.bash $PLOT_DIR_v2/hofx_meanbkg.txt $PLOT_DIR_v2/hofx_meananl.txt


For filbkg=/home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meanbkg.txt and filanl=/home/nonroot/shared-v2/EDU/qgLPF/output/exp_mult_obs/plots/hofx_meananl.txt
RMSD bkg, anl = 1.94588e+07 6.08007e+06



# Experiment 5: LPF Cycling with multiple observations
***

In [31]:
export ncycles=6 # If you want more than 6, you need to extend the truth simulation in 0.qg_tutorial_start!!!

### Make Observations for All Cycles


In [32]:
cd $JEDI_EDU_v2

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   forecast_hour="P$((icyc+15))D"
   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, forecast time = $forecast_hour
   
cat << EOF > ./qgLPF/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: ${curdate}
  filename: /home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.${forecast_hour}.nc
time window: 
  begin: ${curdate_minus6h}
  length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdataout:
        obsfile: /home/nonroot/shared/EDU/qgstart/output/obs/truth_cyc${icyc}.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100                           # Generate 100 observations at random locations
        obs error: 4.0e6                           # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                                     # Generate the observations
obs perturbations: true                            # Add random  measurements to the observations
EOF

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Cycle 1: curdate = 2009-12-31T00:00:00Z, window begin = 2009-12-30T18:00:00Z, forecast time = P16D
Cycle 2: curdate = 2010-01-01T00:00:00Z, window begin = 2009-12-31T18:00:00Z, forecast time = P17D
Cycle 3: curdate = 2010-01-02T00:00:00Z, window begin = 2010-01-01T18:00:00Z, forecast time = P18D
Cycle 4: curdate = 2010-01-03T00:00:00Z, window begin = 2010-01-02T18:00:00Z, forecast time = P19D
Cycle 5: curdate = 2010-01-04T00:00:00Z, window begin = 2010-01-03T18:00:00Z, forecast time = P20D
Cycle 6: curdate = 2010-01-05T00:00:00Z, window begin = 2010-01-04T18:00:00Z, forecast time = P21D


After running, you can verify that there were yaml files created. There should be as many files as there are cycles (set by `ncycles`)

In [33]:
cd $JEDI_EDU_v2
ls ./qgLPF/yamls/makeobs3d_mult_obs_cyc*yaml

./qgLPF/yamls/makeobs3d_mult_obs_cyc1.yaml
./qgLPF/yamls/makeobs3d_mult_obs_cyc2.yaml
./qgLPF/yamls/makeobs3d_mult_obs_cyc3.yaml
./qgLPF/yamls/makeobs3d_mult_obs_cyc4.yaml
./qgLPF/yamls/makeobs3d_mult_obs_cyc5.yaml
./qgLPF/yamls/makeobs3d_mult_obs_cyc6.yaml


We can now run the `qg_hofx` program as in the qgstart tutorial to create observations for each cycle. Note that the outputted cycle observations are saved in the `qgstart/output/obs` directory, as in the qgstart tutorial. For efficiency, the output of the JEDI program has been redirected to log files, with a simple echo statement to confirm each cycle has successfully run or encountered an error.

In [34]:
cd $JEDI_EDU_v2
for icyc in $(seq 1 $ncycles); do
    $JEDI_BIN_v2/qg_hofx3d.x ./qgLPF/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml >& ./qgLPF/output/makeobs_cyc${icyc}.log
    if [ $? -ne 0 ]; then
        echo "ERROR! Something went wrong running for cycle $icyc. Check ./qgLPF/output/makeobs_cyc${icyc}.log"
    else
        echo "SUCCESS run for cycle $icyc"
    fi
done

SUCCESS run for cycle 1
SUCCESS run for cycle 2
SUCCESS run for cycle 3
SUCCESS run for cycle 4
SUCCESS run for cycle 5
SUCCESS run for cycle 6


If you encounter any issues, check the output log created. Otherwise, let's quickly verify observations were created for all cycles with an `ls` command:

In [35]:
cd $JEDI_EDU
ls  ./qgstart/output/obs/truth_cyc*.obs3d.nc

./qgstart/output/obs/truth_cyc1.obs3d.nc
./qgstart/output/obs/truth_cyc2.obs3d.nc
./qgstart/output/obs/truth_cyc3.obs3d.nc
./qgstart/output/obs/truth_cyc4.obs3d.nc
./qgstart/output/obs/truth_cyc5.obs3d.nc
./qgstart/output/obs/truth_cyc6.obs3d.nc


### Construct LPF and forecast yamls

In [36]:
cd $JEDI_EDU_v2
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgLPF/yamls/lpf_template.yaml > ./qgLPF/yamls/lpf_cyc${icyc}.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Cycle 1: curdate = 2009-12-31T00:00:00Z, window begin = 2009-12-30T18:00:00Z, bgfile = /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc
Cycle 2: curdate = 2010-01-01T00:00:00Z, window begin = 2009-12-31T18:00:00Z, bgfile = qgLPF/output/exp_lpf/da/lpf.%mem%.fc.2009-12-31T00:00:00Z.P1D.nc
Cycle 3: curdate = 2010-01-02T00:00:00Z, window begin = 2010-01-01T18:00:00Z, bgfile = qgLPF/output/exp_lpf/da/lpf.%mem%.fc.2010-01-01T00:00:00Z.P1D.nc
Cycle 4: curdate = 2010-01-03T00:00:00Z, window begin = 2010-01-02T18:00:00Z, bgfile = qgLPF/output/exp_lpf/da/lpf.%mem%.fc.2010-01-02T00:00:00Z.P1D.nc
Cycle 5: curdate = 2010-01-04T00:00:00Z, window begin = 2010-01-03T18:00:00Z, bgfile = qgLPF/output/exp_lpf/da/lpf.%mem%.fc.2010-01-03T00:00:00Z.P1D.nc
Cycle 6: curdate = 2010-01-05T00:00:00Z, window begin = 2010-01-04T18:00:00Z, bgfile = qgLPF/output/exp_lpf/da/lpf.%mem%.fc.2010-01-04T00:00:00Z.P1D.nc


In [37]:
cd $JEDI_EDU_v2
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 20); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgLPF/yamls/forecast_template.yaml > ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

Cycle 1: curdate = 2009-12-31T00:00:00Z, forecast_cyc1_mem*.yaml files created!
Cycle 2: curdate = 2010-01-01T00:00:00Z, forecast_cyc2_mem*.yaml files created!
Cycle 3: curdate = 2010-01-02T00:00:00Z, forecast_cyc3_mem*.yaml files created!
Cycle 4: curdate = 2010-01-03T00:00:00Z, forecast_cyc4_mem*.yaml files created!
Cycle 5: curdate = 2010-01-04T00:00:00Z, forecast_cyc5_mem*.yaml files created!
Cycle 6: curdate = 2010-01-05T00:00:00Z, forecast_cyc6_mem*.yaml files created!


### Run the cycling!


In [38]:
cd $JEDI_EDU_v2
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run lpf
   $JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_cyc${icyc}.yaml >& ./qgLPF/output/lpf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running lpf for cycle $icyc. Check ./qgLPF/output/lpf_cyc${icyc}.log"
      break 
   else
      end_time_lpf=`date +%s`
      echo "SUCCESS lpf run for cycle $icyc, took $(( end_time_lpf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 20); do
      $JEDI_BIN_v2/qg_forecast.x ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml >& ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log"
         break 2
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_lpf )) seconds"

done

SUCCESS lpf run for cycle 1, took 92 seconds
SUCCESS forecast run for cycle 1, member 1
SUCCESS forecast run for cycle 1, member 2
SUCCESS forecast run for cycle 1, member 3
SUCCESS forecast run for cycle 1, member 4
SUCCESS forecast run for cycle 1, member 5
SUCCESS forecast run for cycle 1, member 6
SUCCESS forecast run for cycle 1, member 7
SUCCESS forecast run for cycle 1, member 8
SUCCESS forecast run for cycle 1, member 9
SUCCESS forecast run for cycle 1, member 10
SUCCESS forecast run for cycle 1, member 11
SUCCESS forecast run for cycle 1, member 12
SUCCESS forecast run for cycle 1, member 13
SUCCESS forecast run for cycle 1, member 14
SUCCESS forecast run for cycle 1, member 15
SUCCESS forecast run for cycle 1, member 16
SUCCESS forecast run for cycle 1, member 17
SUCCESS forecast run for cycle 1, member 18
SUCCESS forecast run for cycle 1, member 19
SUCCESS forecast run for cycle 1, member 20
SUCCESS ensemble forecast run for cycle 1, took 8 seconds
SUCCESS lpf run for cycle 

Let's look at some of the results produced within the qgLPF/output/exp_lpf/da directory. We will just list out contents from member 1, but you should confirm that 20 members of output were produced in the aforementioned directory.


In [39]:
cd $JEDI_EDU_v2
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ./qgLPF/output/exp_lpf/da/lpf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ./qgLPF/output/exp_lpf/da/lpf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ./qgLPF/output/exp_lpf/da/lpf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ./qgLPF/output/exp_lpf/da/lpf_mean*nc

Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):
./qgLPF/output/exp_lpf/da/lpf.000001.an.2009-12-31T00:00:00Z.nc
./qgLPF/output/exp_lpf/da/lpf.000001.an.2010-01-01T00:00:00Z.nc
./qgLPF/output/exp_lpf/da/lpf.000001.an.2010-01-02T00:00:00Z.nc
./qgLPF/output/exp_lpf/da/lpf.000001.an.2010-01-03T00:00:00Z.nc
./qgLPF/output/exp_lpf/da/lpf.000001.an.2010-01-04T00:00:00Z.nc
./qgLPF/output/exp_lpf/da/lpf.000001.an.2010-01-05T00:00:00Z.nc

Ensemble Member 1 Forecasts (used as backgrounds for each cycle):
./qgLPF/output/exp_lpf/da/lpf.1.fc.2009-12-31T00:00:00Z.P1D.nc
./qgLPF/output/exp_lpf/da/lpf.1.fc.2010-01-01T00:00:00Z.P1D.nc
./qgLPF/output/exp_lpf/da/lpf.1.fc.2010-01-02T00:00:00Z.P1D.nc
./qgLPF/output/exp_lpf/da/lpf.1.fc.2010-01-03T00:00:00Z.P1D.nc
./qgLPF/output/exp_lpf/da/lpf.1.fc.2010-01-04T00:00:00Z.P1D.nc
./qgLPF/output/exp_lpf/da/lpf.1.fc.2010-01-05T00:00:00Z.P1D.nc

Ensemble Mean Analyses:
./qgLPF/output/exp_lpf/da/lpf.000000.an.2009-12-31T00:00:00Z.nc


### Compute RMSE, Plot Sawtooths 

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [40]:
cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU_v2/qgLPF/output/exp_lpf/da/lpf_meanprior.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU_v2/qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

removed '/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt'
Computing rmse for 2009-12-31T00:00:00Z . . .
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
Computing rmse for 2010-01-01T00:00:00Z . . .
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
Computing rmse for 2010-01-02T00:00:00Z . . .
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
Computing rmse for 2010-01-03T00:00:00Z . . .
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
RMSE written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt
Computing rmse for 2010-01-04T00:00:00Z

Next, the python program plot_rmse_sawtooth.py reads in the rmse output file produced by the loop above, parses the information, and plots the sawtooth of background and analysis RMSE for each cycle

Usage:

    python plot_rmse_sawtooth.py <rmse_file.txt> -o <name_of_sawtooth_file.png>



In [41]:
cd $JEDI_EDU_v2/plots_scripts

python plot_rmse_sawtooth.py $JEDI_EDU_v2/qgLPF/output/exp_lpf/plots/rmse_stream.txt \
                             -o $JEDI_EDU_v2/qgLPF/output/exp_lpf/plots/rmse_sawtooth.png

['cyc1', 'cyc2', 'cyc3', 'cyc4', 'cyc5', 'cyc6']
{0: {'bg': [18332770.911122, 21937231.723279, 25336907.136594, 31553325.982597, 38916046.300037, 31747766.766315], 'an': [10832844.55271, 14482980.566012, 17585505.974301, 17132675.967792, 24973475.954119, 20240321.958769]}, 1: {'bg': [20467425.711363, 27248693.821106, 31124174.07458, 37508737.097558, 47712580.050251, 40609200.18158], 'an': [11408623.112094, 18351127.188251, 20285583.538258, 20997141.070156, 29262635.520618, 23247533.080009]}}
Saved plot to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_sawtooth.png


<center><img src="images/rmse_sawtooth.png" alt="sawtooth" width="600"/></center>

Another python script, `compute_ensvar_layers.py`, is provided to compute the 2d-average of ensemble variance for each layer, and output to a text file. It is setup similar to the RMSE computation script.

### Compute Ensemble Variance


In [42]:
cd $JEDI_EDU_v2/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
 for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc"
       bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.an.${curdate}.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgLPF/output/exp_lpf/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN_v2/qg_ens_mean_variance.x ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log"
      break
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
 done #anorbg loop
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done #icyc loop

ls -lhtr ./qgLPF/output/exp_lpf/da/*EnsVariance*nc

SUCCESS qg_ens_variance.x run for cycle 1 bg
SUCCESS qg_ens_variance.x run for cycle 1 an
SUCCESS qg_ens_variance.x run for cycle 2 bg
SUCCESS qg_ens_variance.x run for cycle 2 an
SUCCESS qg_ens_variance.x run for cycle 3 bg
SUCCESS qg_ens_variance.x run for cycle 3 an
SUCCESS qg_ens_variance.x run for cycle 4 bg
SUCCESS qg_ens_variance.x run for cycle 4 an
SUCCESS qg_ens_variance.x run for cycle 5 bg
SUCCESS qg_ens_variance.x run for cycle 5 an
SUCCESS qg_ens_variance.x run for cycle 6 bg
SUCCESS qg_ens_variance.x run for cycle 6 an
-rw-r--r-- 1 nonroot nonroot 76K Apr 24 02:50 ./qgLPF/output/exp_lpf/da/bgEnsVariance.diag.2009-12-31T00:00:00Z.nc
-rw-r--r-- 1 nonroot nonroot 76K Apr 24 02:50 ./qgLPF/output/exp_lpf/da/anEnsVariance.diag.2009-12-31T00:00:00Z.nc
-rw-r--r-- 1 nonroot nonroot 76K Apr 24 02:50 ./qgLPF/output/exp_lpf/da/bgEnsVariance.diag.2010-01-01T00:00:00Z.nc
-rw-r--r-- 1 nonroot nonroot 76K Apr 24 02:50 ./qgLPF/output/exp_lpf/da/anEnsVariance.diag.2010-01-01T00:00:00Z.nc


In [43]:
cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU_v2/qgLPF/output/exp_lpf/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU_v2/qgLPF/output/exp_lpf/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

removed '/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt'
Computing average ensemble variance for 2009-12-31T00:00:00Z . . .
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
Computing average ensemble variance for 2010-01-01T00:00:00Z . . .
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
Computing average ensemble variance for 2010-01-02T00:00:00Z . . .
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
2D-average variance written to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt
Computing average ensemble variance for 2010-01-03T00:00:00Z . . .
2D-average variance writt

In [44]:
cd $JEDI_EDU_v2/plots_scripts

python plot_rmse_sawtooth_withvar.py -m 6e7 $JEDI_EDU_v2/qgLPF/output/exp_lpf/plots/rmse_stream.txt \
                             $JEDI_EDU_v2/qgLPF/output/exp_lpf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU_v2/qgLPF/output/exp_lpf/plots/rmse_sawtooth_withvar.png

Saved plot to /home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_sawtooth_withvar.png


<center><img src="images/rmse_sawtooth_withvar.png" alt="sawtooth_compare" width="600"/></center>